# Beyond Thought Anchors — Reproduce Whitebox Methods

**Method 2**: Receiver Head Attention Analysis  
**Method 3**: Causal Masking (Sentence-level KL Suppression)  

Model: `qwen-14b` = `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` (48 layers, 40 heads)  
Datasets: GPQA (20 questions) + Math (20 questions), correct base CoT only

**Output files:**
- Method 2 GPQA: `csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 2 Math: `csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 3 GPQA: `kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy`
- Method 3 Math: `kl_results/math/qwen-14b/correct/problem_N_kl.npy`

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.__version__)

!nvidia-smi

True
1
2.10.0+cu128
Mon Apr 13 22:06:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [3]:
import os

os.chdir("/content/drive/MyDrive/Beyond-Thought-Anchors/")
print(os.getcwd())

!pip install -r requirements.txt

/content/drive/MyDrive/Beyond-Thought-Anchors
  Preparing metadata (setup.py) ... done
  Created wheel for pkld: filename=pkld-1.2.0-py3-none-any.whl size=8249 sha256=84f3e80bb336122ea3fa46869e3a5d357b63e7f7c4d5ce5bc49c460203464834
  Stored in directory: /root/.cache/pip/wheels/9f/38/98/d2f403cbf6763ab1f9431e62aa18091d214b80ce9fe148da54
Successfully built pkld


---
## Method 2: Receiver Head Attention Analysis

**Step 1** `prep_attn_cache.py` — 加载 14B 模型，对每道题做 **1次** forward pass，一次性提取全部 48层×40头 的 attention 矩阵，存入 `attn_cache/`（1920个 `.npy` 文件/题）。

**Step 2** `generate_rec_csvs.py` — 直接读 `.npy` 缓存，计算 receiver head scores，输出 CSV。不再过模型。

> Step 1 完成后断线重连，Step 2 可直接运行（缓存已在 Drive 上）。

### Method 2 — GPQA

In [4]:
# Step 1: Cache attention matrices for GPQA (1次 forward pass/题)
!python whitebox-analyses/scripts/prep_attn_cache.py \
    --model qwen-14b \
    --dataset gpqa \
    --skip-receiver


Processing model: qwen-14b  dataset: gpqa
Found 20 problems to cache

Caching attention matrices for qwen-14b [gpqa]...
  Model has 48 layers and 40 heads
  Processing 20 problems
Problems:   0% 0/20 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

config.json: 100% 664/664 [00:00<00:00, 4.48MB/s]

tokenizer_config.json: 3.07kB [00:00, 10.7MB/s]

tokenizer.json: 7.03MB [00:00, 138MB/s]
Using float16 for model
Before model loading - GPU Memory: 0.00GB allocated, 0.00GB reserved
`torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors.index.json: 48.0kB [00:00, 136MB/s]


Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]

Fetching 4 files:  25% 1/4 [01:12<03:37, 72.64s/it]

Fetching 4 files: 100% 4/4 [01:19<00:00, 19.96s/it]

Download complete: 100% 29.5G/29.5G [01:19<00:00, 369MB/s]                

Loading weights:   0% 0/579 [00:00<?, ?it/s]
Loading weights:   0% 1/579 [00:

In [ ]:
# Step 2: Generate receiver head CSV for GPQA
# 预计时长：~10s（缓存已就绪）
# 输出：csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py \
    --model-name qwen-14b \
    --data-model qwen-14b \
    --dataset gpqa \
    --top-k 32 \
    --proximity-ignore 16 \
    --output-dir csvs/gpqa

### Method 2 — Math

In [ ]:
# Step 1: Cache attention matrices for Math
!python whitebox-analyses/scripts/prep_attn_cache.py \
    --model qwen-14b \
    --dataset math \
    --skip-receiver

In [ ]:
# Step 2: Generate receiver head CSV for Math
# 输出：csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py \
    --model-name qwen-14b \
    --data-model qwen-14b \
    --dataset math \
    --top-k 32 \
    --proximity-ignore 16 \
    --output-dir csvs/math

---
## Method 3: Causal Masking (KL Suppression)

对每道题的每个句子，mask 掉该句的 attention，重新做 forward pass，计算 token 级 KL 散度，得到句×句因果影响矩阵。

**每道题需要 N_句次 forward pass**（GPQA 平均约 100句/题）。

**输出**: `kl_results/{dataset}/qwen-14b/correct/problem_N_kl.npy`（句×句 numpy 矩阵）

> 行求和 → per-chunk causal importance（Exp 3 方法三排名）。
>
> 断线恢复：`@pkld` 会自动跳过已完成的题，重跑不会重算。

### Method 3 — GPQA

In [ ]:
# 输出：kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py \
    --model-name qwen-14b \
    --dataset gpqa \
    --output-dir kl_results/gpqa

### Method 3 — Math

In [ ]:
# 输出：kl_results/math/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py \
    --model-name qwen-14b \
    --dataset math \
    --output-dir kl_results/math

---
## 验证输出文件

In [ ]:
import os
import numpy as np
import pandas as pd

print("=== Method 2: Receiver Head CSVs ===")
for dataset in ["gpqa", "math"]:
    path = f"csvs/{dataset}/receiver_head_scores_correct_qwen-14b_k32_pi16.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        n_problems = df["problem_number"].nunique()
        print(f"  [{dataset}] {path} — {n_problems} problems, {len(df)} sentences")
    else:
        print(f"  [{dataset}] MISSING: {path}")

print()
print("=== Method 3: KL Suppression Matrices ===")
for dataset in ["gpqa", "math"]:
    kl_dir = f"kl_results/{dataset}/qwen-14b/correct"
    if os.path.exists(kl_dir):
        files = [f for f in os.listdir(kl_dir) if f.endswith(".npy")]
        print(f"  [{dataset}] {kl_dir} — {len(files)} matrices")
        for f in sorted(files)[:3]:
            mat = np.load(os.path.join(kl_dir, f))
            print(f"    {f}: shape={mat.shape}")
    else:
        print(f"  [{dataset}] MISSING: {kl_dir}")